In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
import json
import os
import gc
from tqdm import tqdm

In [ ]:
# ==========================================
# 1. CONFIGURAZIONE PERCORSI E DEVICE
# ==========================================

BASE_JSONL_PATH = "/kaggle/input/datasets/silvioliparoti/couple-generated-v2"
#BASE_MODELS_PATH = "/kaggle/input/datasets/silvioliparoti/deberta-weights-v2-fin"
BASE_MODELS_PATH = "/kaggle/input/datasets/silvioliparoti/deberta-weights-fin-2"

TASKS = [
    {
        "nome": "v2_1 (2 Categorie)",
        "input_jsonl": f"{BASE_JSONL_PATH}/dpo_dataset_ready_v2_1.jsonl",
        "model_path": f"{BASE_MODELS_PATH}/deberta_judge_v2_1.pth",
        "output_jsonl": "dpo_dataset_FILTERED_v2_1.jsonl"
    },
    {
        "nome": "v2_2 (4 Categorie)",
        "input_jsonl": f"{BASE_JSONL_PATH}/dpo_dataset_ready_v2_2.jsonl",
        "model_path": f"{BASE_MODELS_PATH}/deberta_judge_v2_2.pth",
        "output_jsonl": "dpo_dataset_FILTERED_v2_2.jsonl"
    }
]

MODEL_NAME = "microsoft/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device attivo: {DEVICE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
# ==========================================
# 2. DEFINIZIONE ARCHITETTURA MODELLO
# ==========================================
class DebertaJudge(nn.Module):
    def __init__(self):
        super().__init__()
        # Forziamo in float32 per evitare errori di precisione come nello step precedente
        self.bert = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)
        
    def forward(self, input_ids, mask):
        out = self.bert(input_ids, attention_mask=mask)
        return self.regressor(out.last_hidden_state[:, 0, :])

# Funzione di scoring (ora accetta il modello come parametro per usarlo nel ciclo)
def get_score(text, current_model):
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=128, 
        padding=True
    ).to(DEVICE)
    
    with torch.no_grad():
        score = current_model(inputs['input_ids'], inputs['attention_mask'])
    return score.item()

In [ ]:
# ==========================================
# 3. CICLO DI FILTRAGGIO
# ==========================================
for task in TASKS:
    print("\n" + "="*70)
    print(f" INIZIO FILTRAGGIO: {task['nome']}")
    print("="*70)
    
    # Check esistenza file
    if not os.path.exists(task['model_path']):
        print(f"ERRORE: Modello non trovato in {task['model_path']}")
        continue
    if not os.path.exists(task['input_jsonl']):
        print(f"ERRORE: File JSONL non trovato in {task['input_jsonl']}")
        continue

    # 3a. Caricamento Modello Fresco
    print(f"Caricamento pesi da: {os.path.basename(task['model_path'])}...")
    model = DebertaJudge()
    model.load_state_dict(torch.load(task['model_path'], map_location=DEVICE, weights_only=True))
    model.to(DEVICE)
    model.eval()

    # 3b. Esecuzione Filtro
    valid_pairs = 0
    skipped_pairs = 0
    dataset_buffer = []

    with open(task['input_jsonl'], 'r') as f:
        lines = f.readlines()

    print(f"Righe totali da analizzare: {len(lines)}")

    for line in tqdm(lines, desc="Valutazione Battute"):
        try:
            entry = json.loads(line)
            
            # Gestione struttura JSONL
            if isinstance(entry['chosen'], list):
                chosen_text = entry['chosen'][-1]['content']
                rejected_text = entry['rejected'][-1]['content']
            else:
                chosen_text = entry['chosen']
                rejected_text = entry['rejected']
            
            # Calcolo score
            score_chosen = get_score(chosen_text, model)
            score_rejected = get_score(rejected_text, model)
            
            # Criterio di selezione: Originale > Generata + 0.1 di scarto
            if score_chosen > (score_rejected + 0.1):
                dataset_buffer.append(entry)
                valid_pairs += 1
            else:
                skipped_pairs += 1
                
        except Exception as e:
            continue

    # 3c. Salvataggio e Statistiche
    print(f"\n RISULTATI {task['nome']}:")
    print(f" - Scartati: {skipped_pairs}")
    print(f" - TENUTI:   {valid_pairs}")

    with open(task['output_jsonl'], 'w') as f:
        for item in dataset_buffer:
            f.write(json.dumps(item) + "\n")

    print(f" File salvato: {task['output_jsonl']}")

    # 3d. Pulizia Memoria per il prossimo giro
    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\n TUTTE LE VERSIONI SONO STATE FILTRATE CON SUCCESSO!")

In [ ]:
# ==========================================
# 5. ANALISI QUALITATIVA SCARTI (PER LA TESI)
# ==========================================
for task in TASKS:
    print("\n\n" + "=" * 70)
    print(f" CACCIA AGLI ERRORI SU: {task['nome']}")
    print("=" * 70)

    # Ricarico modello per valutare al volo
    model = DebertaJudge()
    model.load_state_dict(torch.load(task['model_path'], map_location=DEVICE, weights_only=True))
    model.to(DEVICE)
    model.eval()

    cattivi_trovati = 0
    with open(task['input_jsonl'], 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if cattivi_trovati >= 5: break # Ci fermiamo dopo 5 esempi

        try:
            entry = json.loads(line)
            
            if isinstance(entry['chosen'], list):
                txt_chosen = entry['chosen'][-1]['content']
                txt_rejected = entry['rejected'][-1]['content']
            else:
                txt_chosen = entry['chosen']
                txt_rejected = entry['rejected']
            
            voto_chosen = get_score(txt_chosen, model)
            voto_rejected = get_score(txt_rejected, model)
            diff = voto_chosen - voto_rejected
            
            # CRITERIO DI SCARTO
            if diff <= 0.1:
                cattivi_trovati += 1
                
                print(f"\n  COPPIA SCARTATA #{cattivi_trovati} (Riga {i})")
                print("-" * 70)
                print(f"CHOSEN (Originale) | Voto Giudice: {voto_chosen:.2f}")
                print(f"\"{txt_chosen[:150]}...\"")
                print("-" * 30)
                print(f"REJECTED (Zephyr)  | Voto Giudice: {voto_rejected:.2f}")
                print(f"\"{txt_rejected[:150]}...\"")
                print("-" * 30)
                
                if voto_rejected > voto_chosen:
                    print(f" PROBLEMA GRAVE: La versione 'Rejected' piace di più al giudice! (Diff: {diff:.2f})")
                else:
                    print(f" PROBLEMA LIEVE: Differenza troppo sottile, dati ambigui. (Diff: {diff:.2f})")
                print("=" * 70)

        except Exception as e:
            continue
            
    # Pulizia memoria
    del model
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ==========================================
# ANALISI QUALITATIVA ACCETTATI (PER LA TESI)
# ==========================================
for task in TASKS:
    print("\n\n" + "=" * 70)
    print(f" ESEMPI DI COPPIE ACCETTATE SU: {task['nome']}")
    print("=" * 70)

    # Ricarico modello per valutare al volo
    model = DebertaJudge()
    model.load_state_dict(torch.load(task['model_path'], map_location=DEVICE, weights_only=True))
    model.to(DEVICE)
    model.eval()

    buoni_trovati = 0
    with open(task['input_jsonl'], 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if buoni_trovati >= 5: break 

        try:
            entry = json.loads(line)
            
            if isinstance(entry['chosen'], list):
                txt_c = entry['chosen'][-1]['content']
                txt_r = entry['rejected'][-1]['content']
            else:
                txt_c = entry['chosen']
                txt_r = entry['rejected']
            
            s_c = get_score(txt_c, model)
            s_r = get_score(txt_r, model)
            
            # Se ACCETTATO
            if s_c > (s_r + 0.1):
                buoni_trovati += 1
                diff = s_c - s_r
                print(f" SUCCESSO #{buoni_trovati} (Riga {i})")
                print(f" Chosen Score:   {s_c:.2f}")
                print(f" Rejected Score: {s_r:.2f}")
                print(f" Margine (Delta): +{diff:.2f}")
                print(f" Chosen: \"{txt_c[:100]}...\"")
                print(f" Reject: \"{txt_r[:100]}...\"")
                print("-" * 40)
        except: 
            continue
            
    # Pulizia memoria
    del model
    gc.collect()
    torch.cuda.empty_cache()

print("\n Analisi terminata per entrambe le versioni!")